# L2: Skill Induction – Trajectory-to-Skill Learning

<p style="background-color:#fff6e4; padding:15px; border-width:3px; border-color:#f5ecda; border-style:solid; border-radius:6px"> ⏳ <b>Note <code>(Kernel Starting)</code>:</b> This notebook takes about 30 seconds to be ready to use. You may start and watch the video while you wait.</p>

In this notebook, you'll turn an agent's traces, its past conversations, tool calls, errors, and fixes, into reusable skills, so tomorrow's agent doesn't repeat yesterday's mistakes.

You'll build a **skill induction pipeline** that:
- Reviews raw traces (what the agent tried, where it failed, what worked) using an LLM
- Drafts a candidate skill
- Runs a **human-in-the-loop review** to approve or reject proposed skills (with a reason)
- Promotes approved skills into the **Skill Box**, a managed collection available to the agent

You'll also see how the induction loop can *enhance* existing skills, comparing a skill's current version against a proposed new version generated from repeated failures, approving the improvement, and re-running the agent to confirm it now succeeds where it previously failed.

In [ ]:
import helper as _lesson_helper
_lesson_helper.ensure_schema()

import importlib
from course_lab import m2_notebook

importlib.reload(m2_notebook)
from course_lab.m2_notebook import connect_stack

mem, conn = connect_stack()

<div style="background-color:#fff6ff; padding:13px; border-width:3px; border-color:#efe6ef; border-style:solid; border-radius:6px">
<p> 💻 &nbsp; <b>Access <code>requirements.txt</code> and <code>helper.py</code> files:</b> 1) click on the <em>"File"</em> option on the top menu of the notebook and then 2) click on <em>"Open"</em>.

<p> ⬇ &nbsp; <b>Download Notebooks:</b> 1) click on the <em>"File"</em> option on the top menu of the notebook and then 2) click on <em>"Download"</em>.</p>
</div>

## Before skill induction

Give the agent a real task using the basic v1 skill, and watch it hit an error. This is the mistake we want it to stop repeating.

In [ ]:
from course_lab.m2_notebook import show_recurring_failure

PAYOFF_TASK, episodes = show_recurring_failure()

## Loading the agents' traces

Load synthetic traces into the database, broken down into conversation turns, tool calls, skill uses, workflow steps, preferences, and errors and fixes.

In [ ]:
from course_lab.coding_trace_synth import LESSON_SKILLS
from course_lab.m2_notebook import print_trace_seed_summary, trace_seed_summary

seed_summary = trace_seed_summary(mem, episodes, LESSON_SKILLS)
print_trace_seed_summary(seed_summary)

## The skill box

Index the active skills and explore the Skill Box before induction. See the skills available to the agent and which one it currently retrieves for the task.

In [ ]:
from course_lab.m2_notebook import (
    build_skill_explorer_app, show_notebook_app, 
    skill_explorer_preview_html,
)
from course_lab.skill_vectorstore import (index_active_skills, 
    search_skill_box)

skill_box = index_active_skills(connection=conn, mem=mem)
before_hit = search_skill_box(skill_box, PAYOFF_TASK, k=1)[0]
skill_app = build_skill_explorer_app(mem)
show_notebook_app(skill_app, skill_explorer_preview_html(mem), 
                  height=525)

## Skill Induction Engine

See the exact contract given to the LLM, then run the induction engine on the traces to surface a repeated failure and propose an enhanced v2 skill, checkpointed and awaiting review.

In [ ]:
from IPython.display import HTML, display
from course_lab.induction_engine import _SYSTEM, induce_all
from course_lab.m2_notebook import (
    engine_contract_html, induced_skill_preview_html, night_outputs,
)

night = night_outputs() 
display(HTML(engine_contract_html(_SYSTEM)))
run_test_episodes = [e for e in episodes if e["topic"] == "run_test_suite"]
proposals = induce_all(run_test_episodes, complete=night)
run_tests_proposal = proposals[0]
display(HTML(induced_skill_preview_html(run_tests_proposal)))

In [ ]:
from course_lab.m2_notebook import run_dream

pending = run_dream(mem, run_test_episodes, complete=night, 
                    connection=conn)
print("OracleSaver checkpointed one candidate update.")
print(f"awaiting review: {pending[0].name} v{pending[0].version}")

## Human in loop

Review the pending proposal, compare it against the current version and approve or reject it with a reason.

In [ ]:
from course_lab.m2_notebook import (
    build_review_app, review_preview_html, show_notebook_app,)

review_app, review_state = build_review_app(pending, mem=mem)
show_notebook_app(
    review_app, 
    review_preview_html(mem, pending, review_state), height=900)

## The skill box with enhanced skill

Re-index the Skill Box and confirm the enhanced skill is now active, replacing the previous version.

In [ ]:
from course_lab.m2_notebook import (
    build_skill_explorer_app, show_notebook_app, 
    skill_explorer_preview_html,
)
from course_lab.skill_vectorstore import (index_active_skills, 
    search_skill_box)

skill_box = index_active_skills(connection=conn, mem=mem)
before_hit = search_skill_box(skill_box, PAYOFF_TASK, k=1)[0]
skill_app = build_skill_explorer_app(mem)
show_notebook_app(skill_app, skill_explorer_preview_html(mem), 
                  height=525)

## Agent with enhanced skill

Search the Skill Box again with the same task and compare what the agent retrieves now versus before, the enhanced skill in action.

In [ ]:
from IPython.display import HTML, display
from course_lab.m2_notebook import retrieval_improvement_html

skill_box = index_active_skills(connection=conn, mem=mem)
after_hit = search_skill_box(skill_box, PAYOFF_TASK, k=1)[0]
display(HTML(retrieval_improvement_html(PAYOFF_TASK, before_hit, 
                                        after_hit)))

## Wrap-up: What You Built

- **Set up the environment**: initialized Oracle Agent Memory (to store traces) and LangGraph Oracle DB (to run the induction loop), and loaded synthetic traces plus five starter skills (`brainstorming`, `requesting-code-review`, `run-the-tests`, `systematic-debugging`, `writing-plans`) into the Skill Box.

- **Saw the "before" state**: ran an agent on a test-suite task using the basic v1 `run-the-tests` skill and watched it hit a real error, the exact kind of repeated mistake skill induction is meant to prevent.

- **Inspected the Induction Engine contract**: the instruction set that tells the LLM how to take episodes from one topic and distill them into steps, tools, errors, fixes, and provenance.

- **Ran the induction loop manually**: the engine reviewed the traces, found a repeated failure and its fix, and proposed an enhanced v2 skill with four refined steps.

- **Exercised the human review gate**: compared v1 vs. v2 side-by-side, then approved the proposal with a reason, promoting v2 to active status in the Skill Box.

- **Confirmed the improvement**: re-ran the agent on the same query; it retrieved v2 of `run-the-tests`, followed the improved procedure, and the test suite passed without failure.

- **Why the human stays in the loop**: approval turns a *proposal* into *behavior* the agent will reuse on every future matching task, high leverage, but also the gate that stops incorrect or even maliciously "planted" traces from becoming permanent agent behavior. Every approval/rejection also gives the induction engine feedback to improve future proposals.